# LLM-Powered Applications & Distributed Computing

## Part 1: Distributed Data Processing with Spark


In [1]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

Done!


### Spark Environment Setup & Data Loading

In [25]:
# Creating a SparkSession 
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("NYC Yellow Taxi Trip Records (January 2024)") \
    .master("local[3]") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

In [26]:
sc = spark.sparkContext  # Access SparkContext
print(sc.getConf().getAll())  # Print all configurations

[('spark.rdd.compress', 'True'), ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'), ('spark.sql.warehouse.dir', 'file:/C:/Users/User/OneDrive/Desktop/COMP3610%20A3/LLM-Powered%20Applications%20and%20Distributed%20Computing/spark-warehouse'), ('spark.master', 'local[3]'), ('spark.executor.memory', '1g'), ('spark.sql.artifact.isolation.enabled', 'false'), ('spark.executor.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add

In [27]:
# Loading the NYC Yellow Taxi Parquet data into a Spark DataFrame
file_path = "data\\raw\\yellow_taxi_data.parquet"
df = spark.read.format("parquet").load(file_path)
df.show(5)
df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [28]:
print(f"Total rows: {df.count()}")
print(f"Total partitions: {df.rdd.getNumPartitions()}")

Total rows: 2964624
Total partitions: 3


In [29]:
# Compare load time for Spark vs Pandas for Parquet file
import pandas as pd
import time
from pathlib import Path

# Used to measure time
def measure_time(func):
    start = time.time()
    result = func()
    end = time.time()
    return result, end - start

parquet_path = Path("data/raw/yellow_taxi_data.parquet")

# Spark load time
def load_spark():
    return spark.read.format("parquet").load(str(parquet_path))

df_spark, spark_time = measure_time(load_spark)
print(f"Spark load time: {spark_time:.2f} seconds")

# Pandas load time
df_pandas, pandas_time = measure_time(lambda: pd.read_parquet(parquet_path))
print(f"Pandas load time: {pandas_time:.2f} seconds")

Spark load time: 0.08 seconds
Pandas load time: 1.16 seconds


Interpretation: Spark usually showed a lower load time than Pandas since it uses lazy evaluation unlike Pandas. So, in other words when Spark's load function is called it builds the plan rather than read the data immediately (only if an action is performed), whereas Pandas reads the entire files into memory right away.

### Data Cleaning & Feature Engineering in Spark

In [ ]:
from pyspark.sql.functions import col

df_nulls = df.filter(
    col("tpep_pickup_datetime").isNull() |
    col("tpep_dropoff_datetime").isNull() |
    col("PULocationID").isNull() |
    col("DOLocationID").isNull() |
    col("fare_amount").isNull() |
    col("trip_distance").isNull()
)
df_nulls.show()

In [36]:
# Remove rows with nulls in critical columns using Spark DataFrame API
critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance"
 ]

df_clean = df.dropna(subset=critical_cols)
print(f"Rows after cleaning: {df_clean.count()}")
print(f"Number of rows removed: {df.count() - df_clean.count()}")

Rows after cleaning: 2964624
Number of rows removed: 0


In [37]:
from pyspark.sql.functions import col

df_nulls = df.filter(
    col("tpep_pickup_datetime").isNull() |
    col("tpep_dropoff_datetime").isNull() |
    col("PULocationID").isNull() |
    col("DOLocationID").isNull() |
    col("fare_amount").isNull() |
    col("trip_distance").isNull()
)
df_nulls.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+---------

In [35]:
# Filter out invalid trips using Spark DataFrame API
from pyspark.sql.functions import col

df_valid = df_clean.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") >= 0) &
    (col("fare_amount") <= 500) &
    (col("tpep_dropoff_datetime") >= col("tpep_pickup_datetime"))
 )
print(f"Rows after filtering invalid trips: {df_valid.count()}")
print(f"Number of rows removed: {df_clean.count() - df_valid.count()}")

Rows after filtering invalid trips: 2870102
Number of rows removed: 94522


In [42]:
# create new column for trip duration in minutes
from pyspark.sql.functions import unix_timestamp, round as spark_round, col

df_enriched = df_valid.withColumn(
    "trip_duration_minutes",
    spark_round(
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60, 2
    )
)
 
df_enriched.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_minutes").show(5)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
| 2024-01-01 00:57:55|  2024-01-01 01:17:43|                 19.8|
| 2024-01-01 00:03:00|  2024-01-01 00:09:36|                  6.6|
| 2024-01-01 00:17:06|  2024-01-01 00:35:01|                17.92|
| 2024-01-01 00:36:38|  2024-01-01 00:44:56|                  8.3|
| 2024-01-01 00:46:51|  2024-01-01 00:52:57|                  6.1|
+--------------------+---------------------+---------------------+
only showing top 5 rows


In [43]:
# create new column for trip speed mph
from pyspark.sql.functions import col, when, round as spark_round

df_enriched = df_enriched.withColumn(
    "trip_speed_mph",
    when(col("trip_duration_minutes") > 0, 
         spark_round((col("trip_distance") / (col("trip_duration_minutes") / 60)), 2)).otherwise(0)
)

df_enriched.select("trip_distance", "trip_duration_minutes", "trip_speed_mph").show(5)

+-------------+---------------------+--------------+
|trip_distance|trip_duration_minutes|trip_speed_mph|
+-------------+---------------------+--------------+
|         1.72|                 19.8|          5.21|
|          1.8|                  6.6|         16.36|
|          4.7|                17.92|         15.74|
|          1.4|                  8.3|         10.12|
|          0.8|                  6.1|          7.87|
+-------------+---------------------+--------------+
only showing top 5 rows


In [104]:
# create a new column for pickup_hour (0-23) and pickup_day_of_week (1=Sunday, 7=Saturday)
from pyspark.sql.functions import hour, dayofweek, col

df_enriched = df_enriched.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))
    )

df_enriched.select("tpep_pickup_datetime", "pickup_hour", "pickup_day_of_week").show(5)

+--------------------+-----------+------------------+
|tpep_pickup_datetime|pickup_hour|pickup_day_of_week|
+--------------------+-----------+------------------+
| 2024-01-01 00:57:55|          0|                 2|
| 2024-01-01 00:03:00|          0|                 2|
| 2024-01-01 00:17:06|          0|                 2|
| 2024-01-01 00:36:38|          0|                 2|
| 2024-01-01 00:46:51|          0|                 2|
+--------------------+-----------+------------------+
only showing top 5 rows


In [105]:
# create a new column for tip_percentage
from pyspark.sql.functions import col, when, round as spark_round

df_enriched = df_enriched.withColumn(
    "tip_percentage",
    when(col("fare_amount") != 0, (col("tip_amount") / col("fare_amount")) * 100)
    .otherwise(0)
)

df_enriched.select("tip_amount", "fare_amount", "tip_percentage").show(5)

+----------+-----------+------------------+
|tip_amount|fare_amount|    tip_percentage|
+----------+-----------+------------------+
|       0.0|       17.7|               0.0|
|      3.75|       10.0|              37.5|
|       3.0|       23.3|12.875536480686694|
|       2.0|       10.0|              20.0|
|       3.2|        7.9| 40.50632911392405|
+----------+-----------+------------------+
only showing top 5 rows
